<a href="https://colab.research.google.com/github/DaniNar2/Aspect-Based-Sentiment-Analysis/blob/main/OTE_Restaurant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing libraries

In [ ]:
import re
import nltk
from nltk.corpus import sentiwordnet as swn
import spacy
nlp = spacy.load("en_core_web_sm")
import pandas as pd
from google.colab import drive
from tqdm import tqdm
tqdm.pandas()

In [ ]:
nltk.download("sentiwordnet")
nltk.download("wordnet")

[nltk_data] Downloading package sentiwordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/sentiwordnet.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Dataset analysis

In [ ]:
drive.mount('/content/drive')
DATA = "/content/drive/MyDrive/SII/Restaurants_Train_v2.csv"
df = pd.read_csv(DATA)

Mounted at /content/drive


In [ ]:
df = df[['Sentence', 'Aspect Term']].copy()
df.columns = ['sentence', 'aspect']
df = df.dropna()
df.head()

,sentence,aspect
0,But the staff was so horrible to us.,staff
1,"To be completely fair, the only redeeming fact...",food
2,"The food is uniformly exceptional, with a very...",food
3,"The food is uniformly exceptional, with a very...",kitchen
4,"The food is uniformly exceptional, with a very...",menu


# Setup and useful functions

In [ ]:
STOP_ADJECTIVES = {
    "first",
    "second",
    "third",
    "last",
    "next",
    "previous",
    "same",
    "other"
}

STOP_VERBS = {
    "be",
    "have",
    "do",
    "find",
    "get",
    "take",
    "make",
    "use",
    "put",
    "go"
}

In [ ]:
# Function to find out if a word might be an opinion word

def is_opinion_word(token):
    lemma = token.lemma_.lower()
    if lemma in STOP_ADJECTIVES:
        return False
    if lemma in STOP_VERBS:
        return False
    synsets = list(swn.senti_synsets(lemma))
    if len(synsets) == 0:
        if token.pos_ == "ADJ":
            return True
        return False
    score = max(
        s.pos_score() + s.neg_score()
        for s in synsets
    )
    return score > 0

In [ ]:
# Function to expand opinion words to opinion phrases

def expand_phrase(token):
    phrase_tokens = []
    for t in token.subtree:
        if t.dep_ in {
            "neg",
            "advmod",
            "amod",
            "acomp"
        }:
            phrase_tokens.append(t)
    for child in token.head.children:
        if child.dep_ == "neg":
            phrase_tokens.append(child)
    if token.dep_ == "acomp":
      has_negation = any(
        child.dep_ == "neg"
        for child in token.head.children
      )
      if has_negation:
        phrase_tokens.append(token.head)
    phrase_tokens.append(token)
    phrase_tokens = sorted(
        set(phrase_tokens),
        key=lambda x: x.i
    )
    return " ".join(t.text for t in phrase_tokens)

In [ ]:
# Function to find out the position of an aspect term in a sentence

def get_aspect_indices(doc, aspect):
    aspect_tokens = [token.text.lower() for token in nlp(aspect)]
    doc_tokens = [token.text.lower() for token in doc]
    for i in range(len(doc_tokens)):
        if doc_tokens[i:i+len(aspect_tokens)] == aspect_tokens:
            return list(range(i, i+len(aspect_tokens)))
    return []

In [ ]:
# Function to calculate appropriate score of metrics

def compute_scores(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0
    )
    return precision, recall, f1

In [ ]:
AUXILIARIES = {
    "is", "are", "was", "were",
    "be", "been", "being",
    "am", "do", "does", "did",
    "have", "has", "had"
}

In [ ]:
# Function to normalize opinions

def normalize_opinion(op):
    op = op.lower()
    op = op.replace("n't", " n't")
    op = re.sub(r"[^\w\s]", "", op)

    tokens = []
    for t in op.split():
        if t in AUXILIARIES:
            continue
        tokens.append(t)
    return set(tokens)

In [ ]:
# Function to split more opinions

def split_opinions(text):
    if pd.isna(text):
        return []
    parts = re.split(r"\||,", str(text))
    return [
        normalize_opinion(p)
        for p in parts
        if p.strip()
    ]

# POS-tagging

In [ ]:
def extract_opinion_pos(sentence, aspect):
    doc = nlp(sentence)
    aspect_idx = get_aspect_indices(doc, aspect)
    if not aspect_idx:
        return ""
    aspect_center = sum(aspect_idx) / len(aspect_idx)
    candidates = []
    for token in doc:
        if token.i in aspect_idx:
            continue
        if token.pos_ not in ["ADJ", "VERB", "NOUN"]:
            continue
        if not is_opinion_word(token):
            continue
        distance = abs(token.i - aspect_center)
        candidates.append(
            (distance, token)
        )
    candidates.sort(key=lambda x: x[0])
    opinions = []
    for _, token in candidates:
        phrase = expand_phrase(token)
        if phrase not in opinions:
            opinions.append(phrase)
    return " | ".join(opinions[:2])

In [ ]:
df["Opinion_POS"] = df.progress_apply(
    lambda row: extract_opinion_pos(
        row["sentence"],
        row["aspect"]
    ),
    axis=1
)

100%|██████████| 3693/3693 [01:10<00:00, 52.53it/s]


# Dependency parsing

In [ ]:
def extract_opinion_dependency(sentence, aspect):
    doc = nlp(sentence)
    aspect_idx = get_aspect_indices(doc, aspect)
    if not aspect_idx:
        return ""
    ranked_opinions = []
    for idx in aspect_idx:
        token = doc[idx]
        candidates = []
        candidates.extend(token.children)
        candidates.append(token.head)
        candidates.extend(token.head.children)
        candidates = list(dict.fromkeys(candidates))
        for cand in candidates:
            if cand.i in aspect_idx:
              continue
            if cand.pos_ not in ["ADJ", "VERB", "ADV"]:
                continue
            if not is_opinion_word(cand):
                continue
            phrase = expand_phrase(cand)
            if len(phrase.split()) > 4:
              continue
            distance = abs(cand.i - token.i)
            ranked_opinions.append(
              (distance, phrase)
            )
    ranked_opinions.sort(
        key=lambda x: x[0]
    )
    opinions = []
    for _, phrase in ranked_opinions:
        if phrase not in opinions:
            opinions.append(phrase)
    return " | ".join(opinions)

In [ ]:
df["Opinion_DEP"] = df.progress_apply(
    lambda row: extract_opinion_dependency(
        row["sentence"],
        row["aspect"]
    ),
    axis=1
)

100%|██████████| 3693/3693 [01:02<00:00, 59.46it/s]


# Evaluation

In [ ]:
# Dataset POS

df_pos = df[df["Opinion_POS"] != ""].copy()
df_pos.reset_index(drop=True, inplace=True)

# Dataset Dependency

df_dep = df[df["Opinion_DEP"] != ""].copy()
df_dep.reset_index(drop=True, inplace=True)

In [ ]:
print("Originale:", len(df))
print("POS:", len(df_pos))
print("Dependency:", len(df_dep))
print("-----")
print("Percentuale POS:", round(len(df_pos)/len(df)*100,2), "%")
print("Percentuale DEP:", round(len(df_dep)/len(df)*100,2), "%")

Originale: 3693
POS: 3613
Dependency: 2373
-----
Percentuale POS: 97.83 %
Percentuale DEP: 64.26 %


In [ ]:
comparison = df[
    [
        "sentence",
        "aspect",
        "Opinion_POS",
        "Opinion_DEP"
    ]
]
comparison.head(50)

,sentence,aspect,Opinion_POS,Opinion_DEP
0,But the staff was so horrible to us.,staff,so horrible,so horrible
1,"To be completely fair, the only redeeming fact...",food,redeeming | only,
2,"The food is uniformly exceptional, with a very...",food,uniformly exceptional | very capable,uniformly exceptional
3,"The food is uniformly exceptional, with a very...",kitchen,very capable | proudly whip,very capable | proudly whip
4,"The food is uniformly exceptional, with a very...",menu,eating | feel,
5,"Not only was the food outstanding, but the lit...",food,outstanding | little,outstanding
6,"Not only was the food outstanding, but the lit...",perks,little | great,little | great
7,Our agreed favorite is the orrechiete with sau...,orrechiete with sausage and chicken,agreed favorite | agreed,
8,Our agreed favorite is the orrechiete with sau...,waiters,kind enough | chicken,kind enough
9,Our agreed favorite is the orrechiete with sau...,meats,half | dish,


# Testing

In [ ]:
GOLD = "/content/drive/MyDrive/SII/gold_standard.csv"
gold = pd.read_csv(GOLD)
gold.head()

,sentence,aspectTerm,opinionPhrase
0,"Late nite omelletes are not good here, there i...",omelletes,not good
1,"Found service above average, but that could be...",service,above average
2,"He not only makes his own homemade mozzarella,...",pie,ultra fresh
3,"Always a nice crowd, but never loud.",crowd,nice
4,The best pad thai i've ever had.,pad thai,best


In [ ]:
gold["pred_pos"] = gold.apply(
    lambda x: extract_opinion_pos(
        x["sentence"],
        x["aspectTerm"]
    ),
    axis=1
)

gold["pred_dep"] = gold.apply(
    lambda x: extract_opinion_dependency(
        x["sentence"],
        x["aspectTerm"]
    ),
    axis=1
)

In [ ]:
gold[
    [
        "sentence",
        "aspectTerm",
        "opinionPhrase",
        "pred_pos",
        "pred_dep"
    ]
].head(20)

,sentence,aspectTerm,opinionPhrase,pred_pos,pred_dep
0,"Late nite omelletes are not good here, there i...",omelletes,not good,Late nite | are not good,Late nite | are not good
1,"Found service above average, but that could be...",service,above average,average,
2,"He not only makes his own homemade mozzarella,...",pie,ultra fresh,ultra | ultra fresh,
3,"Always a nice crowd, but never loud.",crowd,nice,nice | never loud,nice | Always | never loud
4,The best pad thai i've ever had.,pad thai,best,best,best
5,"Even though its good seafood, the prices are t...",prices,too high,too high | good,too high | Even
6,Perhaps this food is considered extreme to an ...,ethnic food,simply dull,actually eaten ethnic | simply dull,actually eaten ethnic | actually
7,"Even if the food wasn't this good, the garden ...",food,wasn’t this good,was n't good | great,Even n't | was n't good
8,"Great bagels, spreads and a good place to hang...",bagels,great,Great | Great spreads good,Great
9,While this is a pretty place in that overly cu...,food,insultingly horrible,overly cute French way | insultingly horrible,insultingly horrible


In [ ]:
correct_pos = (
    gold.apply(
        lambda x:
        str(x["opinionPhrase"]).lower()
        in
        str(x["pred_pos"]).lower(),
        axis=1
    ).sum()
)

correct_dep = (
    gold.apply(
        lambda x:
        str(x["opinionPhrase"]).lower()
        in
        str(x["pred_dep"]).lower(),
        axis=1
    ).sum()
)

print(f"POS Accuracy: {correct_pos}/{len(gold)} = {100*correct_pos/len(gold):.2f}%")
print(f"DEP Accuracy: {correct_dep}/{len(gold)} = {100*correct_dep/len(gold):.2f}%")

POS Accuracy: 68/100 = 68.00%
DEP Accuracy: 56/100 = 56.00%


In [ ]:
tp_pos = fp_pos = fn_pos = 0
tp_dep = fp_dep = fn_dep = 0

for _, row in gold.iterrows():
    gold_ops = split_opinions(row["opinionPhrase"])
    pos_ops = split_opinions(row["pred_pos"])
    dep_ops = split_opinions(row["pred_dep"])

    # POS-tagging
    matched = [False] * len(gold_ops)
    for pred in pos_ops:
        found = False
        for i, gold_op in enumerate(gold_ops):
            if matched[i]:
                continue
            if len(pred & gold_op) > 0:
                tp_pos += 1
                matched[i] = True
                found = True
                break
        if not found:
            fp_pos += 1
    fn_pos += matched.count(False)

    # Dependency parsing
    matched = [False] * len(gold_ops)
    for pred in dep_ops:
        found = False
        for i, gold_op in enumerate(gold_ops):
            if matched[i]:
                continue
            if len(pred & gold_op) > 0:
                tp_dep += 1
                matched[i] = True
                found = True
                break
        if not found:
            fp_dep += 1
    fn_dep += matched.count(False)

In [ ]:
p, r, f = compute_scores(tp_pos, fp_pos, fn_pos)

print("POS")
print("TP:", tp_pos)
print("FP:", fp_pos)
print("FN:", fn_pos)
print("Precision:", round(p,4))
print("Recall:", round(r,4))
print("F1:", round(f,4))

print("-----")

p, r, f = compute_scores(tp_dep, fp_dep, fn_dep)

print("DEP")
print("TP:", tp_dep)
print("FP:", fp_dep)
print("FN:", fn_dep)
print("Precision:", round(p,4))
print("Recall:", round(r,4))
print("F1:", round(f,4))

POS
TP: 89
FP: 100
FN: 16
Precision: 0.4709
Recall: 0.8476
F1: 0.6054
-----
DEP
TP: 71
FP: 45
FN: 34
Precision: 0.6121
Recall: 0.6762
F1: 0.6425
